# S6_02 — MCP Inspector: 브라우저 기반 서버 디버깅

**Skilljar Lesson**: L05 — 서버 인스펙터

## 강의노트 매핑 (`Week_07.md`)

| 절 | 주제 | 라인 |
|---|---|---|
| §1.5 | 서버 인스펙터 — 명령 `mcp dev`, 포트 6277, 편집과 읽기의 검증 연쇄 | 670-763 |

## 사전 준비
본 노트북을 실행하기 전에 다음 조건을 확인한다.
- 본 노트북과 **같은 폴더** 에 `mcp_server.py` 가 있어야 한다 (S6_01 의 §10 셀이 저장).
- 명령 `pip install "mcp[cli]"` 로 SDK 가 설치되어 있어야 한다.
- 브라우저로 주소 `http://localhost:6277` 을 열 수 있어야 한다. 그렇지 못한 환경에서는 §6 의 stdio JSON-RPC 직접 통신을 대체 경로로 사용할 수 있다.

## 학습 목표
본 노트북을 마치면 다음을 할 수 있어야 한다.
1. 명령 `mcp dev mcp_server.py` 로 인스펙터를 기동하고 stdio 서버에 연결한다.
2. Tools, Resources, Prompts 의 세 탭을 둘러보고 UI 안에서 도구를 실행해 본다.
3. 강의노트 §1.5 가 강조하는 **편집한 뒤 다시 읽어 확인하는 검증 연쇄** 를 그대로 재현한다.
4. 브라우저가 막힌 환경 — 예컨대 원격 Jupyter 나 Colab — 에서는 **JSON-RPC 직접 통신** 으로 같은 일을 수행한다.


## §1. MCP 인스펙터란 무엇인가

Python 의 MCP SDK 는 별도 설치 없이 곧바로 쓸 수 있는 **브라우저 기반 인스펙터** 를 함께 제공한다. 이 인스펙터의 역할은 세 가지로 요약된다.

첫째, stdio 방식의 MCP 서버를 자식 프로세스로 띄워 준다. 둘째, 도구를 호출하고 리소스를 조회하며 프롬프트를 렌더링하기 위한 그래픽 사용자 인터페이스를 제공한다. 셋째, 디버깅을 위해 클라이언트와 서버 사이를 오가는 **원본 JSON-RPC 메시지를 그대로** 보여준다.

왜 이것이 중요한가. 인스펙터가 없다면 도구가 올바른 타입을 반환하는지 확인하기 위해서도 서버를 실제 Claude 에 연결해야 한다. 그 과정에는 API 토큰 소모가 따르고, 모델의 응답 시간 때문에 피드백 루프가 느려진다. 강의노트 §1.5 는 이 도구의 가치를 **"LLM 이 보기 전에 서버 계약을 먼저 검증한다"** 라는 한 마디로 정리한다.

> [!finding] 강의노트 §1.5 인용
> 인스펙터는 LLM 없이 서버 계약만 먼저 검증하므로, 도구 스키마, 에러 처리, 반환 타입의 버그를 몇 초 단위 루프로 잡을 수 있다. 그 다음 단계로 Claude 에 붙였을 때, 비로소 "모델이 이 도구를 잘 고르는가" 같은 상위 문제에 집중할 수 있다.


## §2. 셋업 — 같은 폴더에 `mcp_server.py` 가 있는지 확인

이번 노트북은 자체적으로 서버 코드를 정의하지 않는다. 이전 노트북 S6_01 의 §10 셀이 저장한 `mcp_server.py` 를 그대로 사용한다. 따라서 그 파일이 본 노트북과 같은 디렉터리에 존재해야 인스펙터가 그것을 띄워 줄 수 있다. 아래 셀은 단순한 존재 확인일 뿐이지만, 만약 파일이 없다면 S6_01 노트북부터 다시 실행해야 한다.


In [ ]:
import os

server_path = os.path.abspath("mcp_server.py")
print(f"Server path: {server_path}")
print(f"Exists: {os.path.exists(server_path)}")
if os.path.exists(server_path):
    print(f"Size: {os.path.getsize(server_path)} bytes")


## §3. 인스펙터 기동 — 명령 `mcp dev mcp_server.py`

강의노트 §1.5 의 라인 679 부근이 명시하는 표준 명령은 `mcp dev mcp_server.py` 다. 이 명령은 **포트 6277** 에 개발 서버를 띄우고 그것을 가리키는 로컬 URL 을 출력한다. 이 노트북의 인터랙티브 상태를 유지하기 위해, 명령을 Jupyter 셀에서 백그라운드 서브프로세스로 실행한다.

> 참고할 점: 설치된 `mcp[cli]` 의 버전에 따라 포트 번호가 다를 수 있다. CLI 가 시작 시 실제 사용 중인 URL 을 출력하므로, 기대한 주소가 열리지 않는 경우 캡처된 stderr 를 확인한다.


In [ ]:
import subprocess
import shlex
import time

# Inspector 를 백그라운드로 실행
proc = subprocess.Popen(
    shlex.split("mcp dev mcp_server.py"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

print(f"Inspector PID: {proc.pid}")
time.sleep(2)  # URL 출력 시간을 잠깐 준다
print("브라우저에서 열기:  http://localhost:6277")
print("(또는 CLI 출력에 표시된 다른 URL)")


## §4. 인스펙터 화면 빠르게 둘러보기

**Connect** 버튼을 누르면 인스펙터가 stdio 로 서버와 연결되고 네 개의 주요 탭이 활성화된다. 각 탭의 역할은 다음과 같다.

| 탭 이름 | 역할 | 시도해 볼 만한 작업 |
|---|---|---|
| **Tools** | 도구 목록을 조회하고 직접 호출한다 | `read_doc_contents` 와 `edit_document` 호출 |
| **Resources** | 정적 URI 와 템플릿 URI 를 둘러본다 | `docs://documents` 와 `docs://documents/plan.md` (S6_04 에서 다룸) |
| **Prompts** | 서버가 정의한 프롬프트 템플릿을 본다 | 프롬프트 `format` (S6_05 에서 다룸) |
| **Logs** | 원본 JSON-RPC 프레임을 그대로 본다 | 요청과 응답의 식별자가 짝을 이루는지 확인 |


## §5. 편집한 뒤 다시 읽어 확인하는 검증 연쇄

본 절은 강의노트 §1.5 의 라인 720 부근이 제시하는 표준 시나리오다. **Tools** 탭에서 다음 세 단계를 순서대로 수행한다.

**1단계 — 원본 읽기**. 도구 `read_doc_contents` 를 인자 `doc_id="plan.md"` 로 실행한다. 기대하는 결과는 다음과 같다.

> The plan outlines the steps for the project's implementation.

**2단계 — 편집**. 도구 `edit_document` 를 다음 세 인자로 실행한다.
- `doc_id="plan.md"`
- `old_str="outlines"`
- `new_str="describes"`

성공 시 별도의 반환값은 없다. 메서드 `.replace()` 가 메모리 안의 `docs` 딕셔너리를 직접 변경하므로, 응답에는 단지 "성공" 표시만 돌아온다.

**3단계 — 다시 읽기**. 다시 `read_doc_contents("plan.md")` 를 실행한다. 이번에는 다음 결과가 나와야 한다.

> The plan describes the steps for the project's implementation.

이 세 단계가 이루어지는 동안 **Logs** 탭을 함께 보고 있으면, 짝을 이루는 `CallToolRequest` 와 `CallToolResult` 가 차례로 흐르는 것을 확인할 수 있다. 다음 노트북 S6_03 의 `MCPClient` 가 사용할 프로토콜이 바로 이것이다.


## §6. 대체 경로 — stdio 위로 JSON-RPC 를 직접 보낸다

실행 환경이 주소 `localhost:6277` 을 열 수 없는 경우 — 예컨대 Colab 이나 포트 포워딩이 없는 원격 Jupyter 환경 — 인스펙터의 그래픽 인터페이스 대신 **표준 입출력 위로 JSON-RPC 메시지를 직접 주고받아** 같은 결과를 얻을 수 있다. 아래 셀은 핸드셰이크 메서드 `initialize` 를 보내고, 그다음 메서드 `tools/list` 메시지로 도구 카탈로그를 조회하는 과정을 손으로 한 번 따라간다.

이 셀이 보여주는 것이 바로 다음 노트북에서 만들 클래스 `MCPClient` 가 내부에서 대신 처리해 주는 일의 본질이다. 클래스를 쓰지 않고 메시지를 직접 보내 보면, 클라이언트 클래스가 정확히 무엇을 추상화하는지 또렷이 보인다. 직접 손으로 메시지를 만들어 본 경험은, 다음 노트북의 클래스 코드를 읽을 때 "이 메서드가 선로 위로 어떤 메시지를 흘려보내는가" 를 짐작하는 능력을 길러 준다.


In [ ]:
import subprocess
import json as _json

# 서버를 일반 서브프로세스로 띄운다 (Inspector 래퍼 없이)
rpc_proc = subprocess.Popen(
    ["python", "mcp_server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)


def send(msg):
    rpc_proc.stdin.write(_json.dumps(msg) + "\n")
    rpc_proc.stdin.flush()


def recv():
    line = rpc_proc.stdout.readline()
    return _json.loads(line) if line else None


# 1. initialize 핸드셰이크
send({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "notebook", "version": "0.1"},
    },
})
print("initialize ->", recv())

# 2. notifications/initialized (응답 없음)
send({"jsonrpc": "2.0", "method": "notifications/initialized"})

# 3. tools/list — 등록된 도구 카탈로그 조회
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
resp = recv()
if resp and "result" in resp:
    for t in resp["result"]["tools"]:
        print(f"  - {t['name']}: {t['description']}")

rpc_proc.terminate()
rpc_proc.wait(timeout=5)
print("\nJSON-RPC 서브프로세스 종료.")


## §7. 인스펙터 깔끔하게 종료하기

백그라운드 서브프로세스를 시작했으면 반드시 종료까지 책임져야 한다. 종료하지 않으면 포트 6277 이 다음 실행에서도 점유되어 새 인스펙터가 뜨지 못한다. 아래 셀은 정상 종료를 시도하고, 일정 시간 안에 끝나지 않으면 강제 종료한다.


In [ ]:
proc.terminate()
try:
    proc.wait(timeout=5)
    print("Inspector stopped.")
except subprocess.TimeoutExpired:
    proc.kill()
    print("Inspector force-killed.")


## §8. 보너스 — Claude Desktop 또는 Claude Code 에 서버 등록하기

인스펙터에서 모든 테스트를 통과한 서버는 곧장 실제 클라이언트에 꽂아 사용할 수 있다. 가장 자주 쓰는 두 가지 등록 방식은 다음과 같다. Claude Desktop 의 경우 데스크톱 앱이 본 명령으로 서버를 자동 등록하고 시작 시마다 띄워 준다. Claude Code 의 경우 터미널 명령으로 등록하면 같은 머신의 모든 Claude Code 세션에서 그 서버의 도구를 호출할 수 있다.

```bash
# Claude Desktop 에 등록
mcp install mcp_server.py --name DocumentMCP

# Claude Code CLI 에 등록
claude mcp add documentmcp -- python /full/path/to/mcp_server.py
```

강의노트 §2.7 의 Claude Code 스킬 절과, 본 강의 7주차 끝부분의 보너스 자료에서 등록 후의 멀티 에이전트 활용과 Agent SDK 연동 패턴을 함께 다룬다. 인스펙터에서 충분히 검증된 서버만 실제 클라이언트에 등록한다는 원칙을 지키면, 토큰을 낭비하는 디버깅 사이클을 피할 수 있다.


## §9. 다음 단계 안내

- **S6_03 (Client)**: 본 노트북에서 손으로 짠 JSON-RPC 클라이언트를, 깔끔한 async 컨텍스트 매니저 클래스 `MCPClient` 로 다시 만든다.
- **S6_04 (Resources)**: 클라이언트에 `read_resource` 메서드를 추가하고 URI `docs://documents` 와 그 파생 템플릿을 호출한다.
- **S6_05 (Prompts)**: 메서드 `list_prompts` 와 `get_prompt` 를 추가하고, 프롬프트 `format` 을 끝까지 실행한다.
- **`structural/` 트랙**: KDS 조문 데이터를 노출하는 도메인 MCP 서버에 같은 인스펙터 워크플로를 적용해 본다.
